# Whale Fills Analysis
Load all 139 whale fill files into a single pandas DataFrame and explore the data.

In [1]:
import json
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

DATA_DIR = Path("data")

records = []
skipped = []

for f in sorted(DATA_DIR.glob("*.fills.json")):
    address = f.stem.replace(".fills", "")
    with open(f) as fp:
        fills = json.load(fp)
    if not fills:
        skipped.append(address)
        continue
    for fill in fills:
        fill["address"] = address
        records.append(fill)

print(f"Loaded {len(records):,} fill records from {len(records) > 0 and 'multiple' or 0} wallets")
print(f"Skipped {len(skipped)} wallets with no fills")

Loaded 260,565 fill records from multiple wallets
Skipped 3 wallets with no fills


In [2]:
df = pd.DataFrame(records)

# --- Type casting ---
df["px"]         = pd.to_numeric(df["px"],         errors="coerce")  # price
df["sz"]         = pd.to_numeric(df["sz"],         errors="coerce")  # size (asset units)
df["closedPnl"]  = pd.to_numeric(df["closedPnl"],  errors="coerce")  # realized PnL
df["fee"]        = pd.to_numeric(df["fee"],         errors="coerce")  # fee in USDC
df["startPosition"] = pd.to_numeric(df["startPosition"], errors="coerce")  # position before fill

# --- Timestamps ---
df["time"] = pd.to_datetime(df["time"], unit="ms", utc=True)

# --- Derived columns ---
df["notional"] = df["px"] * df["sz"]   # USD value of the fill
df["date"]     = df["time"].dt.date     # plain date for grouping

# --- Rename for clarity ---
df = df.rename(columns={
    "coin":    "asset",
    "px":      "price",
    "sz":      "size",
    "dir":     "direction",
    "side":    "side",       # B=buy A=ask(sell)
    "crossed": "is_taker",
})

# --- Column order ---
cols = ["address", "time", "date", "asset", "direction", "side",
        "price", "size", "notional", "closedPnl", "fee",
        "startPosition", "is_taker", "hash", "oid", "tid", "cloid", "feeToken", "twapId"]
# keep only cols that exist (cloid is absent on some wallets)
cols = [c for c in cols if c in df.columns]
df = df[cols].sort_values("time").reset_index(drop=True)

print(f"DataFrame shape: {df.shape}")
print(f"\nDate range: {df['time'].min().date()} -> {df['time'].max().date()}")
print(f"Unique wallets: {df['address'].nunique()}")
print(f"Unique assets:  {df['asset'].nunique()}")
df.dtypes

DataFrame shape: (260565, 19)

Date range: 2025-01-01 -> 2026-03-02
Unique wallets: 136
Unique assets:  280


address                          str
time             datetime64[ms, UTC]
date                          object
asset                            str
direction                        str
side                             str
price                        float64
size                         float64
notional                     float64
closedPnl                    float64
fee                          float64
startPosition                float64
is_taker                        bool
hash                             str
oid                            int64
tid                            int64
cloid                            str
feeToken                         str
twapId                        object
dtype: object

In [3]:
# Preview the first few rows
df.head(10)

,address,time,date,asset,direction,side,price,size,notional,closedPnl,fee,startPosition,is_taker,hash,oid,tid,cloid,feeToken,twapId
0,0x56cd86d6ef24a3f51ce6992b7f1db751b0a0276a,2025-01-01 00:02:13.189000+00:00,2025-01-01,SOL,Open Short,A,189.85,118.51,22499.1235,0.0,7.874693,-415.03,True,0xb2b70e3879eca0c5652b041a59819401fb00239181c6...,59920411227,178672971303440,NaN,USDC,None
1,0x56cd86d6ef24a3f51ce6992b7f1db751b0a0276a,2025-01-01 00:02:13.189000+00:00,2025-01-01,SOL,Open Short,A,189.84,118.52,22499.8368,0.0,7.874942,-533.54,True,0xb2b70e3879eca0c5652b041a59819401fb00239181c6...,59920411227,241954761246360,NaN,USDC,None
2,0x56cd86d6ef24a3f51ce6992b7f1db751b0a0276a,2025-01-01 00:02:13.189000+00:00,2025-01-01,SOL,Open Short,A,189.85,18.10,3436.2850,0.0,1.202699,-396.93,True,0xb2b70e3879eca0c5652b041a59819401fb00239181c6...,59920411227,1048699953838850,NaN,USDC,None
3,0x56cd86d6ef24a3f51ce6992b7f1db751b0a0276a,2025-01-01 00:02:13.189000+00:00,2025-01-01,SOL,Open Short,A,189.85,200.04,37977.5940,0.0,13.292157,-196.89,True,0xb2b70e3879eca0c5652b041a59819401fb00239181c6...,59920411227,950716203986956,NaN,USDC,None
4,0x56cd86d6ef24a3f51ce6992b7f1db751b0a0276a,2025-01-01 00:02:13.189000+00:00,2025-01-01,SOL,Open Short,A,189.86,69.39,13174.3854,0.0,4.611034,-127.50,True,0xb2b70e3879eca0c5652b041a59819401fb00239181c6...,59920411227,839328278695773,NaN,USDC,None
5,0x56cd86d6ef24a3f51ce6992b7f1db751b0a0276a,2025-01-01 00:02:13.189000+00:00,2025-01-01,SOL,Open Short,A,189.86,118.50,22498.4100,0.0,7.874443,-9.00,True,0xb2b70e3879eca0c5652b041a59819401fb00239181c6...,59920411227,612739870987188,NaN,USDC,None
6,0x56cd86d6ef24a3f51ce6992b7f1db751b0a0276a,2025-01-01 00:02:13.189000+00:00,2025-01-01,SOL,Open Short,A,189.86,9.00,1708.7400,0.0,0.598059,0.00,True,0xb2b70e3879eca0c5652b041a59819401fb00239181c6...,59920411227,330385928165442,NaN,USDC,None
7,0x56cd86d6ef24a3f51ce6992b7f1db751b0a0276a,2025-01-01 00:02:13.189000+00:00,2025-01-01,SOL,Open Short,A,189.84,5.08,964.3872,0.0,0.337535,-652.06,True,0xb2b70e3879eca0c5652b041a59819401fb00239181c6...,59920411227,78775894027613,NaN,USDC,None
8,0x56cd86d6ef24a3f51ce6992b7f1db751b0a0276a,2025-01-01 00:02:13.189000+00:00,2025-01-01,SOL,Open Short,A,189.84,342.86,65088.5424,0.0,22.780989,-657.14,True,0xb2b70e3879eca0c5652b041a59819401fb00239181c6...,59920411227,824649429364843,NaN,USDC,None
9,0x56cd86d6ef24a3f51ce6992b7f1db751b0a0276a,2025-01-01 00:02:39.628000+00:00,2025-01-01,SOL,Open Short,A,190.05,11.67,2217.8835,0.0,0.776259,-1493.26,True,0x605ecb2472107b9f8060041a598339018300632270e9...,59920487962,475027009581528,NaN,USDC,None


In [4]:
# --- Quick summary stats ---
print("=== Direction breakdown ===")
print(df["direction"].value_counts())

print("\n=== Top 15 most traded assets (by fill count) ===")
print(df["asset"].value_counts().head(15))

print("\n=== Top 15 most traded assets (by total notional USD) ===")
print(df.groupby("asset")["notional"].sum().sort_values(ascending=False).head(15).apply(lambda x: f"${x:,.0f}"))

print("\n=== Fills per wallet (top 15) ===")
print(df["address"].value_counts().head(15))

=== Direction breakdown ===
direction
Open Short              70703
Open Long               56132
Close Short             52191
Close Long              36721
Sell                    25287
Buy                     19180
Long > Short              143
Short > Long              139
Spot Dust Conversion       64
Auto-Deleveraging           5
Name: count, dtype: int64

=== Top 15 most traded assets (by fill count) ===
asset
HYPE          35904
ETH           32781
@107          26472
BTC           25669
SOL           13915
ZEC           13013
ASTER          6562
xyz:SILVER     5948
XPL            4653
XRP            3721
@142           3529
LIT            3299
PUMP           3108
xyz:COPPER     3064
xyz:NVDA       2953
Name: count, dtype: int64

=== Top 15 most traded assets (by total notional USD) ===
asset
ETH           $725,216,098
BTC           $623,296,006
SOL           $178,652,307
HYPE           $90,345,289
@107           $69,795,442
ZEC            $44,869,105
@142           $33,361,374

In [ ]:
df["asset"].unique()

<StringArray>
[         'SOL',          'BTC',          'ETH',          '@37',
         'HYPE',         '@107',          'FIL',          'XRP',
     'FARTCOIN',        'AI16Z',
 ...
         'TNSR',           'IO',  'cash:SILVER', 'flx:PLATINUM',
    'cash:TSLA',         '@235',         '@206',         '@234',
    'cash:GOLD',         '@250']
Length: 280, dtype: str

In [6]:
# --- Per-wallet summary ---
import numpy as np

wallet_summary = df.groupby("address").agg(
    fills        = ("tid",       "count"),
    total_volume = ("notional",  "sum"),
    total_pnl    = ("closedPnl", "sum"),
    total_fees   = ("fee",       "sum"),
    assets_traded= ("asset",     "nunique"),
    first_fill   = ("time",      "min"),
    last_fill    = ("time",      "max"),
    open_longs   = ("direction", lambda x: (x == "Open Long").sum()),
    open_shorts  = ("direction", lambda x: (x == "Open Short").sum()),
).sort_values("total_volume", ascending=False).reset_index()

# Compute long_pct safely (avoid pd.NA which breaks round())
den = wallet_summary["open_longs"] + wallet_summary["open_shorts"]
# convert denominator zeros to np.nan so division yields NaN (float) not pd.NA
ratio = wallet_summary["open_longs"].astype(float).div(den.replace(0, np.nan))
wallet_summary["long_pct"] = (ratio * 100).round(1)

wallet_summary["total_volume"] = wallet_summary["total_volume"].apply(lambda x: f"${x:,.0f}")
wallet_summary["total_pnl"]    = wallet_summary["total_pnl"].apply(lambda x: f"${x:,.0f}")
wallet_summary["total_fees"]   = wallet_summary["total_fees"].apply(lambda x: f"${x:,.0f}")

print("Per-wallet summary (top 20 by volume):")
wallet_summary.head(20)

Per-wallet summary (top 20 by volume):


,address,fills,total_volume,total_pnl,total_fees,assets_traded,first_fill,last_fill,open_longs,open_shorts,long_pct
0,0x9b83f16d0a6456f90a8a330f04c0ca1b2f0425b0,2000,"$131,786,127","$11,513","$58,852",1,2025-09-13 04:10:09.670000+00:00,2025-09-13 14:10:04.964000+00:00,278,953,22.6
1,0xe7ec7fbf4f195fc8e57d814e15c3a2857cb632a3,2000,"$110,773,824","$-461,371","$39,794",2,2025-10-15 07:08:06.246000+00:00,2025-10-18 03:54:03.078000+00:00,1190,0,100.0
2,0xd62d484bda5391d75b414e68f9ddcedb207b7d91,2000,"$98,763,916","$-1,202,471","$40,488",1,2025-06-27 21:24:37.278000+00:00,2025-10-02 18:57:19.172000+00:00,0,1396,0.0
3,0x091159a8106b077c13e89bc09701117e8b5f129a,2000,"$94,005,680","$14,719,018","$24,515",2,2025-09-12 22:31:29.200000+00:00,2025-10-12 05:47:50.692000+00:00,0,1567,0.0
4,0xa5b0edf6b55128e0ddae8e51ac538c3188401d41,2000,"$83,383,747","$60,263","$23,828",1,2025-11-21 09:24:02.156000+00:00,2025-11-21 12:41:37.936000+00:00,1854,0,100.0
5,0x4ed0b41dff79e0e53a054903873899bc32abc853,1762,"$77,910,605","$-967,982","$16,147",1,2026-02-02 08:25:39.241000+00:00,2026-02-25 12:41:50.877000+00:00,1208,0,100.0
6,0x77eeda199553e33b246e4b4666849b9ad0972902,2000,"$65,474,634","$-642,059","$25,833",9,2025-09-17 19:33:04.832000+00:00,2025-10-07 17:01:10.922000+00:00,780,264,74.7
7,0x7eb9026d0183942458bd4660371c3cc0a2df3beb,2000,"$55,615,774","$1,587,928","$19,345",3,2025-10-23 20:54:18.999000+00:00,2025-11-13 23:07:42.875000+00:00,0,1224,0.0
8,0x74da055d4ccf9dca3f15789c603e1a88a1d2d81f,2000,"$52,561,239","$288,817","$9,663",2,2026-01-17 05:32:24.884000+00:00,2026-01-19 17:04:15.085000+00:00,382,891,30.0
9,0x6859da14835424957a1e6b397d8026b1d9ff7e1e,2000,"$51,220,481","$-451,130",$-463,23,2026-02-13 22:01:13.713000+00:00,2026-02-15 00:02:49.623000+00:00,1,1117,0.1
